<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/simple_langchain_rag_colab_demo-11thapr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple LangChain + OpenAI + Chroma RAG Demo in Google Colab

This notebook is a **beginner-friendly, line-by-line demo** for AI residents.

It shows how to:

1. Load API keys from **Google Colab Secrets**
2. Upload a few **PDF files**
3. Read and split the PDFs into chunks
4. Create embeddings with **OpenAI**
5. Store vectors in **ChromaDB**
6. Run **RAG** over the uploaded files
7. Add **web search** using **SERPAPI**
8. Add a very simple **memory** component so the chatbot remembers earlier context

---

## Colab secrets you should create first

In the **Secrets** panel in Colab, create these secrets and turn on notebook access:

- `OPENAI_API_KEY`
- `SERPAPI_API_KEY`

This notebook keeps things intentionally simple and heavily commented so you can teach from it directly.

In [1]:
# Install the packages we need.
# We keep the list small and focused on the demo.

!pip -q install -U langchain langchain-community langchain-openai langchain-chroma chromadb pypdf google-search-results

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━

## 1) Imports

We import only the core pieces we need:
- file loading
- chunking
- embeddings
- vector database
- chat model
- prompt templates
- simple memory
- SERPAPI search

In [2]:
import os
import shutil
from pathlib import Path

from google.colab import files, userdata

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

from langchain_community.utilities import SerpAPIWrapper

## 2) Load API keys from Google Colab secrets

This is cleaner than pasting keys directly into the notebook.

In [3]:
# Read secrets from the Colab Secrets manager and place them in environment variables.
# Make sure you created both secrets in the left-side Secrets panel.

os.environ["OPENAI_API_KEY"] = userdata.get("openai")
os.environ["SERPAPI_API_KEY"] = userdata.get("SERPAPI_API_KEY")

print("OPENAI key loaded:", bool(os.environ.get("OPENAI_API_KEY")))
print("SERPAPI key loaded:", bool(os.environ.get("SERPAPI_API_KEY")))

OPENAI key loaded: True
SERPAPI key loaded: True


## 3) Create a folder and upload PDFs

This cell opens a file picker in Colab.
Upload a few PDFs and we will save them into `/content/data`.

You can re-run this cell any time to add more files.

In [4]:
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

for file_name, file_bytes in uploaded.items():
    save_path = DATA_DIR / file_name
    with open(save_path, "wb") as f:
        f.write(file_bytes)

print("Files currently in data folder:")
for p in DATA_DIR.iterdir():
    print("-", p.name)

Saving DDS_Employee_Handbook_Synthetic_v1 (2).pdf to DDS_Employee_Handbook_Synthetic_v1 (2).pdf
Files currently in data folder:
- DDS_Employee_Handbook_Synthetic_v1 (2).pdf


## 4) Load the PDF files

`PyPDFDirectoryLoader` reads all PDFs inside a folder.

Each page usually becomes a `Document` object with text plus metadata.

In [5]:
loader = PyPDFDirectoryLoader(str(DATA_DIR))
docs = loader.load()

print("Number of loaded document pages:", len(docs))
print("\nSample metadata from the first loaded page:")
print(docs[0].metadata if docs else "No documents found.")

Number of loaded document pages: 4

Sample metadata from the first loaded page:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-03T08:04:57+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-03-03T08:04:57+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': '/content/data/DDS_Employee_Handbook_Synthetic_v1 (2).pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}


## 5) Split the documents into chunks

LLMs and vector stores usually work better on smaller chunks than on entire PDF pages.

We use `RecursiveCharacterTextSplitter`, which is a common default choice.
- `chunk_size=1000` means each chunk is about 1000 characters
- `chunk_overlap=200` keeps some repeated context between chunks

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(docs)

print("Number of chunks:", len(chunks))
print("\nPreview of the first chunk:")
print(chunks[0].page_content[:800] if chunks else "No chunks found.")

Number of chunks: 11

Preview of the first chunk:
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This handbook explains workplace expectations, benefits, and policies. If any local law conflicts with
this handbook, applicable law prevails.
2. Company Values & Ways of Working
 Build with clarity: define


## 6) Create embeddings and store them in ChromaDB

We use:
- `OpenAIEmbeddings` to convert text chunks into vectors
- `Chroma` to store those vectors locally on disk

The `persist_directory` lets us save the vector database so we can reuse it later.

In [7]:
# If you want a clean rebuild every time you run the notebook, remove the old database folder first.
CHROMA_DIR = "/content/chroma_db"

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    collection_name="demo_rag_collection",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

# Add the chunks to the vector database.
vectorstore.add_documents(chunks)

print("Chunks indexed into Chroma.")
print("Chroma data saved at:", CHROMA_DIR)

Chunks indexed into Chroma.
Chroma data saved at: /content/chroma_db


## 7) Run a simple similarity search

Before building the full RAG flow, it is useful to show students how retrieval works directly.

In [8]:
query = "What are the most important topics in these documents?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Retrieved chunk {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:700])


--- Retrieved chunk 1 ---
Metadata: {'creator': 'anonymous', 'page': 1, 'keywords': '', 'title': 'untitled', 'author': 'anonymous', 'creationdate': '2026-03-03T08:04:57+00:00', 'producer': 'ReportLab PDF Library - (opensource)', 'subject': 'unspecified', 'total_pages': 4, 'moddate': '2026-03-03T08:04:57+00:00', 'source': '/content/data/DDS_Employee_Handbook_Synthetic_v1 (2).pdf', 'page_label': '2', 'trapped': '/False'}
 Step 3: Review outcomes; further action may include role change or termination (per contract/law).
10. Learning, Training & Development
DDS invests in learning because it is core to our mission.
 Team members may receive access to internal courses, recordings, and selected external trainings.
 Sharing learnings is encouraged: short notes, demos, internal talks.
 Training time is scheduled with managers to protect delivery commitments.
11. Confidentiality & Intellectual Property
Confidential information includes (but is not limited to): learner data, pricing, partne

## 8) Create a retriever

A retriever is the part that fetches the most relevant chunks for a user question.

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

## 9) Create the chat model

We use a simple OpenAI chat model for the answer generation step.

In [10]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

## 10) Build a simple RAG prompt

The prompt uses:
- the retrieved document context
- chat history
- the user's current question

This lets us demonstrate both **RAG** and **memory** in one place.

In [11]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful assistant for a beginner RAG demo.

Use the retrieved context to answer the user's question.
If the answer is not in the context, say that clearly.
Keep the answer simple and grounded in the retrieved context.
""",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        """
Question:
{input}

Retrieved context:
{context}
""",
    ),
])

## 11) Helper function to format retrieved documents

This converts the retrieved chunks into one text block for the prompt.

In [12]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## 12) Create a simple RAG chain with memory

We will:
1. retrieve relevant chunks
2. format them
3. pass them to the prompt
4. get the final answer from the LLM

Then we wrap the chain with `RunnableWithMessageHistory` so it remembers prior turns.

This is a very simple in-memory demo.  
It will remember context **while the notebook session is alive**.

In [13]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Step A: Build a small chain that first retrieves relevant chunks.
retrieval_step = {
    "input": itemgetter("input"),
    "chat_history": itemgetter("chat_history"),
    "context": itemgetter("input") | retriever | RunnableLambda(format_docs),
}

# Step B: Send everything into the prompt, model, and string output parser.
rag_chain = retrieval_step | rag_prompt | llm | StrOutputParser()

# Step C: Store chat history in a Python dictionary.
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Step D: Wrap the chain with message history.
rag_chain_with_memory = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

## 13) Ask questions and see memory in action

Try asking:
- a question about the uploaded PDFs
- then a follow-up question like "Can you summarize that in 3 bullets?"
- then "What was my previous question about?"

Because we attached memory, the model can use the prior conversation.

In [14]:
session_id = "demo-user-1"

question_1 = "Give me a simple summary of the uploaded documents."
answer_1 = rag_chain_with_memory.invoke(
    {"input": question_1},
    config={"configurable": {"session_id": session_id}}
)

print("Q1:", question_1)
print("\nA1:", answer_1)

Q1: Give me a simple summary of the uploaded documents.

A1: The uploaded document is a synthetic employee handbook for Decoding Data Science (DDS), effective from March 3, 2026. It outlines the company's focus on data science, AI, and applied generative AI, emphasizing a global mindset and a culture of trust. The handbook includes workplace expectations, company values, and policies regarding data protection and privacy. It also notes that the handbook may be updated and that local labor laws take precedence over its content. Additionally, it serves as a training tool and is not intended as legal advice.


In [15]:
question_2 = "Now give me that summary in 3 short bullet points."
answer_2 = rag_chain_with_memory.invoke(
    {"input": question_2},
    config={"configurable": {"session_id": session_id}}
)

print("Q2:", question_2)
print("\nA2:", answer_2)

Q2: Now give me that summary in 3 short bullet points.

A2: - DDS promotes a culture of respect, inclusion, and continuous learning, emphasizing data integrity and confidentiality.
- Employment classifications include full-time, part-time, and contractors, with specific terms outlined in offer letters.
- The company is committed to a discrimination-free workplace and encourages reporting of harassment concerns confidentially.


In [16]:
# This lets students inspect exactly what was saved in memory.
history = get_session_history(session_id)

print("Stored messages in memory:")
for msg in history.messages:
    print(f"\n{msg.type.upper()}:")
    print(msg.content)

Stored messages in memory:

HUMAN:
Give me a simple summary of the uploaded documents.

AI:
The uploaded document is a synthetic employee handbook for Decoding Data Science (DDS), effective from March 3, 2026. It outlines the company's focus on data science, AI, and applied generative AI, emphasizing a global mindset and a culture of trust. The handbook includes workplace expectations, company values, and policies regarding data protection and privacy. It also notes that the handbook may be updated and that local labor laws take precedence over its content. Additionally, it serves as a training tool and is not intended as legal advice.

HUMAN:
Now give me that summary in 3 short bullet points.

AI:
- DDS promotes a culture of respect, inclusion, and continuous learning, emphasizing data integrity and confidentiality.
- Employment classifications include full-time, part-time, and contractors, with specific terms outlined in offer letters.
- The company is committed to a discrimination-f

## 14) Add web search with SERPAPI

Now we add a simple search tool.

This is useful when:
- the uploaded PDFs do not contain the answer
- the user asks for something recent or external
- you want to compare internal document answers with web results

In [17]:
search = SerpAPIWrapper()

web_result = search.run("Latest news about LangChain")
print(web_result[:2000])

['March 2026: LangChain Newsletter. It feels like spring has sprung here, and so has a new NVIDIA integration, ticket sales for Interrupt 2026, and announcing ...', 'Announcements ; Introducing Insights Agent & Multi-turn Evals, new features in LangSmith! 1, 330, October 23, 2025 ; We launched 1.0 versions of LangChain and ...', 'LangChain Announces Enterprise Agentic AI Platform Built with NVIDIA · What the Platform Delivers · Build with LangGraph, Deep Agents, and AI-Q: ...', 'LangChain provides the engineering platform and open source frameworks developers use to build, test, and deploy reliable AI agents.', "We've renamed LangSmith Agent Builder to LangSmith Fleet . What changed Agent Builder is now LangSmith Fleet . You'll see the new name reflected across the...", 'r/LangChain: LangChain is an open-source framework and developer toolkit that helps developers get LLM applications from prototype to production. It…', 'We surveyed over 1,300 professionals — from engineers and product

## 15) A very simple hybrid function: use docs + optional web search

This helper does the following:
- runs retrieval on the local PDFs
- optionally runs SERPAPI search
- combines both into one prompt
- keeps memory

This is still intentionally simple and easy to teach.

In [18]:
hybrid_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful assistant in a simple beginner demo.

Use the local document context first.
Use web search results only as an extra source when they are provided.
If something is missing from the documents, say so clearly.
""",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        """
Question:
{input}

Local document context:
{context}

Optional web search results:
{web_context}
""",
    ),
])

hybrid_chain = (
    {
        "input": itemgetter("input"),
        "chat_history": itemgetter("chat_history"),
        "context": itemgetter("input") | retriever | RunnableLambda(format_docs),
        "web_context": itemgetter("web_context"),
    }
    | hybrid_prompt
    | llm
    | StrOutputParser()
)

hybrid_chain_with_memory = RunnableWithMessageHistory(
    hybrid_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [19]:
def ask_hybrid(question, use_web=False, session_id="demo-user-2"):
    web_context = ""

    if use_web:
        web_context = search.run(question)

    answer = hybrid_chain_with_memory.invoke(
        {
            "input": question,
            "web_context": web_context,
        },
        config={"configurable": {"session_id": session_id}},
    )

    return answer

## 16) Test the hybrid assistant

Use `use_web=False` for pure document RAG.  
Use `use_web=True` when you want external search results included.

In [20]:
print(ask_hybrid("What are the main ideas discussed in my uploaded PDFs?", use_web=False))

The main ideas discussed in your uploaded PDF, which appears to be a synthetic employee handbook for Decoding Data Science (DDS), include:

1. **Company Overview**: DDS is a Dubai-based academy and consulting practice focused on data science and AI, emphasizing a global mindset and a high-trust culture.

2. **Learning, Training & Development**: DDS prioritizes learning and development, providing access to internal and external training resources. Team members are encouraged to share their learnings and schedule training time with managers.

3. **Confidentiality & Intellectual Property**: The document outlines the importance of maintaining confidentiality regarding sensitive information, such as learner data and internal documents. It also addresses the ownership of work created during employment and the respect for third-party intellectual property.

4. **Data Protection & Privacy**: DDS is committed to handling personal and business data responsibly, adhering to data protection and pr

In [21]:
print(ask_hybrid("What are some recent updates about Decoding Data Science?", use_web=True))

The recent updates about Decoding Data Science (DDS) include:

1. **Effective Date of Handbook**: The current employee handbook is effective as of March 3, 2026, indicating that it has been recently updated to reflect the latest policies and practices.

2. **Focus on Data Science and AI**: DDS continues to emphasize its role as an academy and consulting practice in data science and AI, with a commitment to practical outcomes and a supportive community.

3. **Policy Governance**: The handbook states that it may be updated as DDS evolves, with HR responsible for communicating any material changes. The latest version of the handbook is available in the internal knowledge base.

4. **Emerging Trends**: According to external sources, there is a notable shift in the field of AI and data science from experimentation to execution, with trends such as generative AI, automated machine learning (AutoML), and edge computing reshaping practices in the industry.

5. **AI Residency Program**: DDS is 

## 17) Save and reload the vector database later

Because we used a `persist_directory`, we can reconnect to the same Chroma database later without rebuilding it from scratch.

This is helpful in Colab demos when you want to explain indexing once, then reuse the stored vectors.

In [22]:
reloaded_vectorstore = Chroma(
    collection_name="demo_rag_collection",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

reloaded_results = reloaded_vectorstore.similarity_search(
    "Give me a summary of the documents",
    k=2
)

for i, doc in enumerate(reloaded_results, start=1):
    print(f"\nReloaded result {i}:")
    print(doc.page_content[:600])


Reloaded result 1:
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This ha

Reloaded result 2:
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
18. Policy Governance & Updates
This handbook may be updated as DDS evolves.
 HR communicates material changes.
 The latest version lives in the internal knowledge base.
 Your contract and local labor law remain the governing documents where applicable.
Page 4
Decoding Data

## 18) Teaching notes

### What this notebook demonstrates
- **LangChain loaders** for PDFs
- **Chunking**
- **OpenAI embeddings**
- **Chroma vector database**
- **Retriever-based RAG**
- **SERPAPI search**
- **Simple memory with chat history**

### What this notebook does *not* try to do
- advanced agents
- tool calling orchestration
- citations with structured sources
- production-grade memory
- evaluation
- UI deployment

That is intentional. This notebook is meant to be the **smallest clean teaching demo**.

## 19) Suggested next upgrades for residents

After students understand this notebook, the next upgrades can be:
1. Add a small **Gradio** or **Streamlit** UI
2. Add **source citations**
3. Add support for more file types
4. Add **query rewriting**
5. Add **conversation-aware retrieval**
6. Add **LangSmith tracing**
7. Add **better memory storage** beyond notebook runtime